# 01 - Build Task A and Task B Datasets

This notebook does all preprocessing and creates the two datasets you need for the project:

- **Task A (baseline / reconstruction):** predict winner using same-map statistics
- **Task B (main project):** predict winner using **past-only** information available before the map starts

## What this notebook saves
Inside `OUTPUT_DIR`, it will save:

- `taskA_dataset.csv`
- `taskB_dataset.csv`
- `dataset_summary.json`

## Before you run
1. Upload `NewVLRDataRaw.csv` to Colab when prompted, **or** put it in your Google Drive and set `RAW_CSV_PATH` manually.
2. Run every cell in order.

In [ ]:
# Optional: mount Google Drive so files persist
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/ds4400_valorant'
else:
    PROJECT_DIR = '/content/ds4400_valorant'

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('OUTPUT_DIR  =', OUTPUT_DIR)

In [ ]:
# Option A: upload the raw CSV directly in Colab
UPLOAD_FILE = True

RAW_CSV_PATH = None

if UPLOAD_FILE:
    from google.colab import files
    uploaded = files.upload()
    RAW_CSV_PATH = list(uploaded.keys())[0]

# Option B: if you already placed the file in Drive, set RAW_CSV_PATH manually:
# RAW_CSV_PATH = '/content/drive/MyDrive/ds4400_valorant/NewVLRDataRaw.csv'

print('RAW_CSV_PATH =', RAW_CSV_PATH)

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)

In [ ]:
# Load raw data
raw = pd.read_csv(RAW_CSV_PATH)
print(raw.shape)
raw.head()

## 1) Basic cleaning and target creation

We:
- convert `Date` to datetime
- remove the single broken 1970 row
- remove draws / non-standard incomplete rows
- create the target `Team1 Win`
- remove rows with invalid team IDs

In [ ]:
raw['Date'] = pd.to_datetime(raw['Date'], errors='coerce')

# Keep valid years only
raw = raw[raw['Date'].dt.year >= 2020].copy()

# Remove weird invalid team IDs
raw = raw[(raw['Team1ID'] > 0) & (raw['Team2ID'] > 0)].copy()

# Create target
raw['Team1 Win'] = (raw['Team1 Rounds'] > raw['Team2 Rounds']).astype(int)

# Remove draws / weird partial rows
raw = raw[
    ~(
        (raw['Team1 Rounds'] == raw['Team2 Rounds']) |
        ((raw['Team1 Rounds'] < 13) & (raw['Team2 Rounds'] < 13))
    )
].copy()

# Sort chronologically for all past-only features
raw = raw.sort_values(['Date', 'GameID']).reset_index(drop=True)

print(raw.shape)
print(raw[['Date', 'GameID', 'Team1 Name', 'Team2 Name', 'Team1 Rounds', 'Team2 Rounds', 'Team1 Win']].head())

## 2) Train/test split

We will use a time-based split:
- train = 2020 to 2023
- test = 2024

This makes the main Task B setup much more honest because it avoids using future matches when learning from the past.

In [ ]:
SPLIT_DATE = pd.Timestamp('2024-01-01')
raw['Split'] = np.where(raw['Date'] < SPLIT_DATE, 'train', 'test')

print(raw['Split'].value_counts())
print(raw.groupby(raw['Date'].dt.year)['Split'].value_counts())

## 3) Task A dataset (baseline)

This is your **same-map baseline**.
It intentionally uses same-map statistics, but we still drop the most obvious direct-leak columns:

- `Team1 Rounds`, `Team2 Rounds`
- attack/defense round splits
- direct delta summary columns
- text/link columns and IDs that are not useful predictors

In [ ]:
taskA_df = raw.copy()

drop_taskA = [
    'MatchID', 'GameID', 'EventID', 'Date', 'Team1ID', 'Team2ID',
    'Team1 Name', 'Team2 Name', 'Round Breakdown', 'VOD Link',
    'Team1 Win',
    # direct scoreboard leakage
    'Team1 Rounds', 'Team2 Rounds',
    'Team1 Atk Rounds', 'Team2 Atk Rounds',
    'Team1 Def Rounds', 'Team2 Def Rounds',
    # direct summary leakage
    'Team1 DeltaK/D', 'Team2 DeltaK/D',
    'Team1 DeltaFK/FD', 'Team2 DeltaFK/FD',
    # prior-game id columns are for Task B only
    'Team1Game1', 'Team1Game2', 'Team1Game3', 'Team1Game4', 'Team1Game5',
    'Team2Game1', 'Team2Game2', 'Team2Game3', 'Team2Game4', 'Team2Game5',
    'Split'
]

taskA_X = taskA_df.drop(columns=drop_taskA, errors='ignore').copy()
taskA_y = taskA_df['Team1 Win'].copy()

taskA_dataset = pd.concat(
    [
        raw[['Date', 'Split', 'Team1 Win']].reset_index(drop=True),
        taskA_X.reset_index(drop=True)
    ],
    axis=1
)

print(taskA_dataset.shape)
taskA_dataset.head()

## 4) Build Elo ratings for Task B

In [ ]:
def add_elo_features(df, k=20, base_rating=1500):
    ratings = {}
    team1_pre = np.empty(len(df))
    team2_pre = np.empty(len(df))

    for i, row in enumerate(df.itertuples(index=False)):
        t1 = row.Team1ID
        t2 = row.Team2ID
        r1 = ratings.get(t1, base_rating)
        r2 = ratings.get(t2, base_rating)

        team1_pre[i] = r1
        team2_pre[i] = r2

        expected_t1 = 1.0 / (1.0 + 10 ** ((r2 - r1) / 400.0))
        actual_t1 = getattr(row, 'Team1_Win')

        ratings[t1] = r1 + k * (actual_t1 - expected_t1)
        ratings[t2] = r2 + k * ((1 - actual_t1) - (1 - expected_t1))

    out = df.copy()
    out['Team1 Elo'] = team1_pre
    out['Team2 Elo'] = team2_pre
    out['Elo Diff'] = out['Team1 Elo'] - out['Team2 Elo']
    return out

raw_elo = raw.copy()
raw_elo.columns = [c.replace(' ', '_') for c in raw_elo.columns]
raw_elo = add_elo_features(raw_elo, k=20, base_rating=1500)

raw_elo[['Team1ID', 'Team2ID', 'Team1_Elo', 'Team2_Elo', 'Elo_Diff']].head()

## 5) Build long-format team history table

For every map we create:
- one row from Team 1's perspective
- one row from Team 2's perspective

Then we compute **past-only rolling features** within each team:
- recent win rate
- recent rating
- recent ACS
- recent KAST
- recent ADR
- recent kills
- recent deaths
- overall prior win rate
- map-specific prior win rate

In [ ]:
team_stats = ['Rating', 'ACS', 'KAST', 'ADR', 'Kills', 'Deaths']

t1 = pd.DataFrame({
    'row_id': raw_elo.index,
    'Date': raw_elo['Date'],
    'GameID': raw_elo['GameID'],
    'Map': raw_elo['Map'],
    'TeamID': raw_elo['Team1ID'],
    'OpponentID': raw_elo['Team2ID'],
    'side': 't1',
    'won': raw_elo['Team1_Win'].astype(int),
    'Rating': raw_elo['Team1_Rating'],
    'ACS': raw_elo['Team1_ACS'],
    'KAST': raw_elo['Team1_KAST'],
    'ADR': raw_elo['Team1_ADR'],
    'Kills': raw_elo['Team1_Kills'],
    'Deaths': raw_elo['Team1_Deaths'],
})

t2 = pd.DataFrame({
    'row_id': raw_elo.index,
    'Date': raw_elo['Date'],
    'GameID': raw_elo['GameID'],
    'Map': raw_elo['Map'],
    'TeamID': raw_elo['Team2ID'],
    'OpponentID': raw_elo['Team1ID'],
    'side': 't2',
    'won': 1 - raw_elo['Team1_Win'].astype(int),
    'Rating': raw_elo['Team2_Rating'],
    'ACS': raw_elo['Team2_ACS'],
    'KAST': raw_elo['Team2_KAST'],
    'ADR': raw_elo['Team2_ADR'],
    'Kills': raw_elo['Team2_Kills'],
    'Deaths': raw_elo['Team2_Deaths'],
})

long_df = pd.concat([t1, t2], ignore_index=True)
long_df = long_df.sort_values(['TeamID', 'Date', 'GameID']).reset_index(drop=True)

rolling_features = ['won', 'Rating', 'ACS', 'KAST', 'ADR', 'Kills', 'Deaths']
for col in rolling_features:
    long_df[f'{col}_last3'] = (
        long_df.groupby('TeamID')[col]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    )

long_df['team_prev_games'] = long_df.groupby('TeamID').cumcount()
long_df['team_prev_wins'] = long_df.groupby('TeamID')['won'].cumsum() - long_df['won']
long_df['team_prev_winrate'] = np.where(
    long_df['team_prev_games'] > 0,
    long_df['team_prev_wins'] / long_df['team_prev_games'],
    np.nan
)

long_df['team_map_prev_games'] = long_df.groupby(['TeamID', 'Map']).cumcount()
long_df['team_map_prev_wins'] = long_df.groupby(['TeamID', 'Map'])['won'].cumsum() - long_df['won']
long_df['team_map_prev_winrate'] = np.where(
    long_df['team_map_prev_games'] > 0,
    long_df['team_map_prev_wins'] / long_df['team_map_prev_games'],
    np.nan
)

long_df.head()

## 6) Head-to-head past-only features

In [ ]:
taskB_base = raw_elo.copy()

lower_team = np.minimum(taskB_base['Team1ID'], taskB_base['Team2ID'])
upper_team = np.maximum(taskB_base['Team1ID'], taskB_base['Team2ID'])

taskB_base['pair_key'] = lower_team.astype(str) + '_' + upper_team.astype(str)
taskB_base['lower_team_win'] = np.where(
    taskB_base['Team1ID'] == lower_team,
    taskB_base['Team1_Win'],
    1 - taskB_base['Team1_Win']
)

taskB_base['pair_prev_games'] = taskB_base.groupby('pair_key').cumcount()
taskB_base['pair_lower_prev_wins'] = (
    taskB_base.groupby('pair_key')['lower_team_win'].cumsum() - taskB_base['lower_team_win']
)

taskB_base['Team1 H2H Prev Wins'] = np.where(
    taskB_base['Team1ID'] == lower_team,
    taskB_base['pair_lower_prev_wins'],
    taskB_base['pair_prev_games'] - taskB_base['pair_lower_prev_wins']
)
taskB_base['Team2 H2H Prev Wins'] = taskB_base['pair_prev_games'] - taskB_base['Team1 H2H Prev Wins']

taskB_base['Team1 H2H Prev WinRate'] = np.where(
    taskB_base['pair_prev_games'] > 0,
    taskB_base['Team1 H2H Prev Wins'] / taskB_base['pair_prev_games'],
    np.nan
)
taskB_base['Team2 H2H Prev WinRate'] = np.where(
    taskB_base['pair_prev_games'] > 0,
    taskB_base['Team2 H2H Prev Wins'] / taskB_base['pair_prev_games'],
    np.nan
)

taskB_base[['pair_prev_games', 'Team1 H2H Prev WinRate', 'Team2 H2H Prev WinRate']].head()

## 7) Merge team-history features back to match-level rows

In [ ]:
feature_map = {
    'won_last3': 'Recent WinRate 3',
    'Rating_last3': 'Recent Rating 3',
    'ACS_last3': 'Recent ACS 3',
    'KAST_last3': 'Recent KAST 3',
    'ADR_last3': 'Recent ADR 3',
    'Kills_last3': 'Recent Kills 3',
    'Deaths_last3': 'Recent Deaths 3',
    'team_prev_games': 'Prior Games',
    'team_prev_winrate': 'Prior WinRate',
    'team_map_prev_games': 'Map Prior Games',
    'team_map_prev_winrate': 'Map Prior WinRate',
}

team_hist_cols = ['row_id'] + list(feature_map.keys())

t1_hist = long_df[long_df['side'] == 't1'][team_hist_cols].copy()
t2_hist = long_df[long_df['side'] == 't2'][team_hist_cols].copy()

t1_hist = t1_hist.rename(columns={'row_id': 'row_id', **{k: f'Team1 {v}' for k, v in feature_map.items()}})
t2_hist = t2_hist.rename(columns={'row_id': 'row_id', **{k: f'Team2 {v}' for k, v in feature_map.items()}})

taskB_dataset = (
    taskB_base
    .merge(t1_hist, left_index=True, right_on='row_id', how='left')
    .merge(t2_hist, on='row_id', how='left')
)

print(taskB_dataset.shape)
taskB_dataset.head()

## 8) Keep only past-only Task B features + create difference features

In [ ]:
# Human-readable columns
taskB_dataset.columns = [c.replace('_', ' ') for c in taskB_dataset.columns]

# Difference features
diff_pairs = [
    ('Team1 Elo', 'Team2 Elo', 'Elo Diff'),
    ('Team1 Recent WinRate 3', 'Team2 Recent WinRate 3', 'Recent WinRate 3 Diff'),
    ('Team1 Recent Rating 3', 'Team2 Recent Rating 3', 'Recent Rating 3 Diff'),
    ('Team1 Recent ACS 3', 'Team2 Recent ACS 3', 'Recent ACS 3 Diff'),
    ('Team1 Recent KAST 3', 'Team2 Recent KAST 3', 'Recent KAST 3 Diff'),
    ('Team1 Recent ADR 3', 'Team2 Recent ADR 3', 'Recent ADR 3 Diff'),
    ('Team1 Recent Kills 3', 'Team2 Recent Kills 3', 'Recent Kills 3 Diff'),
    ('Team1 Recent Deaths 3', 'Team2 Recent Deaths 3', 'Recent Deaths 3 Diff'),
    ('Team1 Prior WinRate', 'Team2 Prior WinRate', 'Prior WinRate Diff'),
    ('Team1 Map Prior WinRate', 'Team2 Map Prior WinRate', 'Map Prior WinRate Diff'),
    ('Team1 H2H Prev WinRate', 'Team2 H2H Prev WinRate', 'H2H Prior WinRate Diff')
]

for c1, c2, new_c in diff_pairs:
    taskB_dataset[new_c] = taskB_dataset[c1] - taskB_dataset[c2]

keep_taskB = [
    'Date', 'Split', 'Team1 Win',
    'Map', 'Series Odds', 'Team1 Map Odds',
    'Team1 Elo', 'Team2 Elo', 'Elo Diff',
    'Team1 Recent WinRate 3', 'Team2 Recent WinRate 3', 'Recent WinRate 3 Diff',
    'Team1 Recent Rating 3', 'Team2 Recent Rating 3', 'Recent Rating 3 Diff',
    'Team1 Recent ACS 3', 'Team2 Recent ACS 3', 'Recent ACS 3 Diff',
    'Team1 Recent KAST 3', 'Team2 Recent KAST 3', 'Recent KAST 3 Diff',
    'Team1 Recent ADR 3', 'Team2 Recent ADR 3', 'Recent ADR 3 Diff',
    'Team1 Recent Kills 3', 'Team2 Recent Kills 3', 'Recent Kills 3 Diff',
    'Team1 Recent Deaths 3', 'Team2 Recent Deaths 3', 'Recent Deaths 3 Diff',
    'Team1 Prior Games', 'Team2 Prior Games',
    'Team1 Prior WinRate', 'Team2 Prior WinRate', 'Prior WinRate Diff',
    'Team1 Map Prior Games', 'Team2 Map Prior Games',
    'Team1 Map Prior WinRate', 'Team2 Map Prior WinRate', 'Map Prior WinRate Diff',
    'pair prev games',
    'Team1 H2H Prev WinRate', 'Team2 H2H Prev WinRate', 'H2H Prior WinRate Diff'
]

cols_available = [c for c in keep_taskB if c in taskB_dataset.columns]
taskB_dataset = taskB_dataset[cols_available].copy()

print(taskB_dataset.shape)
taskB_dataset.head()

## 9) Quick sanity checks

In [ ]:
print('Task A shape:', taskA_dataset.shape)
print('Task B shape:', taskB_dataset.shape)

print('\nTask A missingness (top 15):')
print(taskA_dataset.isna().sum().sort_values(ascending=False).head(15))

print('\nTask B missingness (top 15):')
print(taskB_dataset.isna().sum().sort_values(ascending=False).head(15))

In [ ]:
summary = {
    'raw_rows_after_cleaning': int(raw.shape[0]),
    'taskA_rows': int(taskA_dataset.shape[0]),
    'taskA_cols': int(taskA_dataset.shape[1]),
    'taskB_rows': int(taskB_dataset.shape[0]),
    'taskB_cols': int(taskB_dataset.shape[1]),
    'train_rows': int((raw['Split'] == 'train').sum()),
    'test_rows': int((raw['Split'] == 'test').sum()),
    'class_balance_team1_win': float(raw['Team1 Win'].mean())
}
summary

## 10) Save outputs

In [ ]:
taskA_path = os.path.join(OUTPUT_DIR, 'taskA_dataset.csv')
taskB_path = os.path.join(OUTPUT_DIR, 'taskB_dataset.csv')
summary_path = os.path.join(OUTPUT_DIR, 'dataset_summary.json')

taskA_dataset.to_csv(taskA_path, index=False)
taskB_dataset.to_csv(taskB_path, index=False)

with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
print(taskA_path)
print(taskB_path)
print(summary_path)

## 11) EDA Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

raw['Team1 Win'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Class Balance: Team1 Win')
axes[0].set_xlabel('Team1 Win')
axes[0].set_ylabel('Count')

raw['Date'].dt.year.value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Matches by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figure1_class_balance_and_years.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(
    taskB_dataset.select_dtypes(include=[np.number]).drop(columns=['Team1 Win']).corr().iloc[:15, :15],
    cmap='coolwarm',
    center=0
)
plt.title('Task B Correlation Heatmap (first 15 numeric features)')
plt.savefig(os.path.join(OUTPUT_DIR, 'figure2_taskB_correlation_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()